# Quickstart: Getting Started with `qox` 

`qox` is a quantitative finance library designed for high-performance option pricing. This guide covers basic valuation, market setup, and how the engine handles different exercise styles and numerical methods automatically.


In [ ]:
!pip install qox -q

## 1. Environment Setup

We define our valuation context using `zoneinfo` for time-zone awareness.


In [ ]:
import time
from datetime import datetime
from zoneinfo import ZoneInfo
import qox

ny_tz = ZoneInfo("America/New_York")
valuation_time = datetime(2025, 9, 25, 17, 0, tzinfo=ny_tz)
expiry = datetime(2026, 9, 25, 17, 0, tzinfo=ny_tz)

market_frame = qox.OptionMarketFrame(
    spot=95.0,
    rate_curve=qox.RateCurve.continuous(0.05, qox.DayCountConvention.Act365Fixed),
    vol_surface=qox.VolSurface.flat(0.2, qox.DayCountConvention.Act365Fixed),
)


## 2. Smart Engine Selection

One of the core features of `qox` is its ability to select the most efficient pricing engine based on the instrument's characteristics:

| Instrument | Default Engine | Reason |
| :--- | :--- | :--- |
| **European (Call/Put)** | Analytic | Exact closed-form solution. |
| **American Call** | Analytic | Optimal for non-dividend stocks. |
| **American Put** | FDM | Handles early exercise boundary. |

### Example: American Call (Analytic)


In [ ]:
american_call = qox.VanillaOption(100.0, expiry, qox.OptionType.Call, qox.ExerciseStyle.American)

call_result = american_call.valuation().at(valuation_time).market(market_frame).compute()
print(f"American Call Price: {call_result.price:.4f} (Calculated via Analytic Formula)")


### Example: American Put (Automatic FDM)
When pricing an American Put, `qox` detects that no analytic solution exists and automatically switches to the Finite Difference Method (FDM).


In [ ]:
american_put = qox.VanillaOption(100.0, expiry, qox.OptionType.Put, qox.ExerciseStyle.American)

put_result = american_put.valuation().at(valuation_time).market(market_frame).compute()
print(f"American Put Price: {put_result.price:.4f} (Calculated via FDM)")


## 3. Customizing Numerical Methods

While `qox` provides sensible defaults, you can explicitly tune the numerical engine (e.g., changing the number of nodes or time steps) using a `Config` policy.


In [ ]:
# Configure high-precision FDM
fdm_config = qox.FdmConfig(nodes=500, time_steps=5, grid_std_devs=6.0)
config = qox.Config().add_policy(qox.InstrumentPolicy().american().put().fdm(fdm_config))

tuned_result = (
    american_put.valuation()
    .at(valuation_time)
    .market(market_frame)
    .config(config)
    .compute()
)

print(f"Tuned Price: {tuned_result.price:.4f}")
print("-" * 20)
print("Greeks (FDM):")
print(f"  Delta: {tuned_result.greeks.delta:.5f}")
print(f"  Gamma: {tuned_result.greeks.gamma:.5f}")
print(f"  Theta: {tuned_result.greeks.theta:.5f}")
